# Compare Missing Images

Try to find missing images in the Fitzpatrick dataset in the SkinCAP dataset

In [ ]:
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from PIL import Image
import requests
from huggingface_hub import hf_hub_download
from dotenv import load_dotenv
import os

DATA_DIR = Path("../data/")
DATA_PREP_DIR = Path("../data_preprocessed/")

SCINCAP_META_PATH = DATA_DIR / "skincap_v240623.csv"
FITSPATRICK_META_PATH = DATA_DIR / "fitzpatrick17k.csv"
FITZPATRICK_FAILS = DATA_PREP_DIR / 'fitzpatrick_failed.csv'

HF_SECRET_PATH = "/Users/nicholasyousefi/Documents/Coding/School/Harvard/Fall_2025/ac215/project/other_secrets/huggingface_read_repos.txt"
load_dotenv(HF_SECRET_PATH)

/Users/nicholasyousefi/Google Drive Private (nicholas.yousefi.26@gmail.com)/School/Harvard/Classes/Spring 2026/bmi712/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
skincap_meta = pd.read_csv(SCINCAP_META_PATH)
skincap_meta.columns

Index(['id', 'skincap_file_path', 'ori_file_path', 'disease', 'caption_zh',
       'caption_zh_polish', 'caption_zh_polish_en', 'remark', 'source',
       'skin_tone', 'malignant', 'fitzpatrick_scale', 'fitzpatrick_centaur',
       'nine_partition_label', 'three_partition_label', 'url', 'Vesicle',
       'Papule', 'Macule', 'Plaque', 'Abscess', 'Pustule', 'Bulla', 'Patch',
       'Nodule', 'Ulcer', 'Crust', 'Erosion', 'Excoriation', 'Atrophy',
       'Exudate', 'Purpura/Petechiae', 'Fissure', 'Induration', 'Xerosis',
       'Telangiectasia', 'Scale', 'Scar', 'Friable', 'Sclerosis',
       'Pedunculated', 'Exophytic/Fungating', 'Warty/Papillomatous',
       'Dome-shaped', 'Flat topped', 'Brown(Hyperpigmentation)', 'Translucent',
       'White(Hypopigmentation)', 'Purple', 'Yellow', 'Black', 'Erythema',
       'Comedo', 'Lichenification', 'Blue', 'Umbilicated', 'Poikiloderma',
       'Salmon', 'Wheal', 'Acuminate', 'Burrow', 'Gray', 'Pigmented', 'Cyst',
       'Do not consider this image

In [3]:
skincap_meta_simple = skincap_meta.loc[
    skincap_meta['source'] == 'fitzpatrick17k',
    ['id', 'skincap_file_path', 'ori_file_path']
]
skincap_meta_simple['hash'] = skincap_meta_simple['ori_file_path'].apply(lambda x: Path(x).stem)
skincap_meta_simple.head()

,id,skincap_file_path,ori_file_path,hash
655,656,656.png,fa2911a9b13b6f8af79cb700937cc14f.jpg,fa2911a9b13b6f8af79cb700937cc14f
656,657,657.png,e702b1a7dc40aa1d8e85ccdb019c4ab2.jpg,e702b1a7dc40aa1d8e85ccdb019c4ab2
657,658,658.png,8438db40abd1eccfbc7ee4b469f1b6f1.jpg,8438db40abd1eccfbc7ee4b469f1b6f1
658,659,659.png,9a3af1bc39e115bcc6931170cf8a00bb.jpg,9a3af1bc39e115bcc6931170cf8a00bb
659,660,660.png,59ccb668671950ca657b6bc48213b763.jpg,59ccb668671950ca657b6bc48213b763


In [4]:
# get the images in skincap that failed to download
fitz_failed = pd.read_csv(FITZPATRICK_FAILS)

# failed fitzpatrick images that are present in skincap
in_skincap_flg = fitz_failed['hash'].isin(skincap_meta_simple['hash'])
fitz_failed[in_skincap_flg].shape[0]

2811

In [5]:
# Login using e.g. `huggingface-cli login` to access this dataset
# ds = load_dataset("joshuachou/SkinCAP")

In [6]:
# ds['train'][3999]['image'].show()

In [7]:
fitz_meta = pd.read_csv(FITSPATRICK_META_PATH)
completed_in_skincap_flg = fitz_meta['md5hash'].isin(skincap_meta_simple['hash']) & ~fitz_meta['md5hash'].isin(fitz_failed['hash'])
fitz_meta[completed_in_skincap_flg].head()

,md5hash,fitzpatrick_scale,fitzpatrick_centaur,label,nine_partition_label,three_partition_label,qc,url,url_alphanum
12689,26a6a7a544d39a116e49d848040ec79d,5,4,mucinosis,inflammatory,non-neoplastic,NaN,http://atlasdermatologico.com.br/img?imageId=8259,httpwwwatlasdermatologicocombrimgimageId8259.jpg
12693,fdfd34f8f20abd710bf9583de1084f2b,4,2,nematode infection,inflammatory,non-neoplastic,NaN,http://atlasdermatologico.com.br/img?imageId=3214,httpwwwatlasdermatologicocombrimgimageId3214.jpg
12694,1069ab14d1badfb5de1c814b70403753,5,4,erythema nodosum,inflammatory,non-neoplastic,NaN,http://atlasdermatologico.com.br/img?imageId=1981,httpwwwatlasdermatologicocombrimgimageId1981.jpg
12702,b5a9aa5155e59feee65f731641bcbe6d,6,6,pityriasis rosea,inflammatory,non-neoplastic,NaN,http://atlasdermatologico.com.br/img?imageId=5587,httpwwwatlasdermatologicocombrimgimageId5587.jpg
12705,68528c33db3fc238d183b5d2c14d6f75,3,3,melanoma,malignant melanoma,malignant,NaN,http://atlasdermatologico.com.br/img?imageId=4244,httpwwwatlasdermatologicocombrimgimageId4244.jpg


In [8]:
filename = skincap_meta_simple.loc[skincap_meta_simple['hash'] == '26a6a7a544d39a116e49d848040ec79d','skincap_file_path'].iloc[0]
print(filename)

3460.png


In [9]:
image_path_hf = "kincap/{filename}"
local_path = hf_hub_download(
    repo_id="joshuachou/SkinCAP",
    filename=image_path_hf.format(filename=filename),
    repo_type="dataset",
    token=os.environ.get("HF_TOKEN"),  # or omit if public and you’re logged in via cache
)
print(local_path)

RemoteEntryNotFoundError: 404 Client Error. (Request ID: Root=1-69d67d9d-200dbbb20a73ec75299b032b;1a282ee0-2469-442c-9e22-6f666693ef36)

Entry Not Found for url: https://huggingface.co/datasets/joshuachou/SkinCAP/resolve/main/kincap/3460.png.